In [1]:
import pickle
import re 
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate,  PauliEvolutionGate, QAOAAnsatz


from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_precompute import CommutingGateRouterPrecompute
from qiskit_qaoa.utils.commuting_gate_router_precompute_rzz import CommutingGateRouterPrecomputeRzz

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph


In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N2_W2.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
# ess = ExtendedSwapStrategy.from_grid(n, T)
ess = ExtendedSwapStrategy.from_all_to_all(n * T)

num_physical_qubits = ess._num_vertices
max_layers = 0

Keeping constraints at times: [0]


In [4]:
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterPrecompute(
            ess,
            max_layers=max_layers,
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [5]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

Max layers needed to apply swap decompose: 0
Could not find shortest sequence by BFS, try heuristic
Gates we cannot directly implement: 0
[]
Transpiling un-implemented gates


In [6]:
tqc.draw(fold=-1)

global phase: 0.34569
         ┌───────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  
q_0 -> 0 ┤ Rz(8.875) ├──────────────────────────────────────────────────────────────────■────────■──────────────────────────────────────────────────────────■────────■────────────■────────■──────────────────────────────────────────────────────────────────■────────■────────────────────────────────────────────────■───────────────────────────────────────────────■─────────────────────────────────■───────────────────────■────■────────────────────────────────────────────────────■─────────────────────────────────────■─────────────────────────────────────■─────────────────────────────────────■──────────────────────────────────────────────────────────────────────■──
         └───────────┘                                                                  │        │                                                          │        │            │        │                                                                  │        │                                                │                                               │                                 │                       │    │                                                    │                                   ┌─┴─┐                                 ┌─┴─┐                                 ┌─┴─┐                                                                  ┌─┴─┐
q_1 -> 1 ──────■───────────────────────■────────────────────────────────────────────────┼────────┼──────────────────────────────────────────────────────────┼────────┼────────────┼────────┼────────■─────────■───────────────────────────────────────────────┼────────┼────────────────────────────────────────────────┼───────────────────────────────────────────────┼─────────■───────────────────────┼───────────────────────┼────┼────■───────────────────────────────────────────────┼────■───────────────────────────■──┤ X ├──■───────────────────────────■──┤ X ├──■───────────────────────────■──┤ X ├──■───────────────────────────■────■───────────────────────────■──┤ X ├
             ┌─┴─┐     ┌───────────┐   │                                                │      ┌─┴─┐    ┌───────────┐                                       │        │          ┌─┴─┐      │      ┌─┴─┐       │                                               │        │                                              ┌─┴─┐                                           ┌─┴─┐       │                       │                       │  ┌─┴─┐  │                                             ┌─┴─┐  │                           │  └───┘  │                           │  └───┘  │                           │  └───┘  │                           │  ┌─┴─┐                       ┌─┴─┐└───┘
q_2 -> 2 ────┤ X ├─────┤ Rz(0.625) ├───┼────────■───────────────────────────────────────┼──────┤ X ├────┤ Rz(0.625) ├──■────────────────────────────────────┼────────┼──────────┤ X ├──────┼──────┤ X ├───────┼────■──────────────────────────────────────────┼────────┼──────────────■────────────────────────────■──┤ X ├──■─────────────────────────────────────■──┤ X ├──■────┼───────────────────────┼──────────────────■────┼──┤ X ├──┼────■─────────────────────────────────────■──┤ X ├──┼───────────────────────────┼─────────┼───────────────────────────┼─────────┼───────────────────────────┼─────────┼───────────────────────────┼──┤ X ├──■─────────────────■─

In [7]:
print_circuit_info(tqc, 'TQC')

TQC has 6 qubits
TQC has 71 non-local gates and 57 non-local depth
TQC contains ['cx', 'rz'] gates.


In [8]:
pm_rzz = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterPrecomputeRzz(
            ess,
            max_layers=max_layers,
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [9]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc_rzz = pm_rzz.run(qc)

Max layers needed to apply swap decompose: 0
Rz interaction: (1, 2)
Rz interaction: (4, 5)
Rz interaction: (0, 3)
Could not find shortest sequence by BFS, try heuristic
Could not find shortest sequence by heuristic
Could not find shortest sequence by BFS, try heuristic
Could not find shortest sequence by heuristic
Could not find shortest sequence by BFS, try heuristic
Could not find shortest sequence by heuristic
Could not find shortest sequence by BFS, try heuristic
Could not find shortest sequence by heuristic
Rz interaction: (1, 5)
Could not find shortest sequence by BFS, try heuristic
Could not find shortest sequence by heuristic
Gates we cannot directly implement: 0
[]
Transpiling un-implemented gates


In [10]:
tqc_rzz.draw(fold=-1)

global phase: 0.34569
         ┌───────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                           
q_0 -> 0 ┤ Rz(8.875) ├────────────────────────────■────────────────────────────■──────────────────────────────────────────■────────────────────────■─────────────────■─────────────────────────────────────────────────────────────────■───────────────────────■────────────────────────────────────────────────────────────────────────■───────────────────────────────────────■──────────────────────────────────────────────────────────────────■────■────────────────■───────
         └───────────┘                            │                            │                                          │                        │                 │                                                                 │                       │                                                                        │                                       │                                                                  │    │                │       
q_1 -> 1 ──────■──────────────────────■───────────┼────────────■───────────────┼──────────────────────────────────────────┼────────────────────────┼─────────────────┼─────────────────────────────────────────────────────────────────┼────────■─────────■────┼────────■───────────■───────────────────────────────────────────────────┼─────────────■─────────────────────────┼────────────────────────────────────────────────■─────────────────┼────┼────────────────┼────■──
             ┌─┴─┐     ┌───────────┐  │           │ZZ(0.625)   │               │                                          │                        │                 │                                                                 │        │       ┌─┴─┐  │        │           │                                                   │             │                       ┌─┴─┐                                              │               ┌─┴─┐  │                │    │  
q_2 -> 2 ────┤ X ├─────┤ Rz(0.625) ├──┼───────────■────────────┼───■───────────┼────────────■─────────────────────────■───┼────────────────────────┼─────────────────┼───────────■───────────────────────■─────────────────────────────┼────────┼────■──┤ X ├──┼────────┼───────────┼───────────■────────────■──────────────────────────┼─────────────┼───────────■───────────┤ X ├─■─────────────■──────────────────────────────┼────────■──────┤ X ├──┼────────────────┼────┼──
         ┌───┴───┴────┐└───────────┘  │                        │   │ZZ(0.625)  │            │                         │   │                      ┌─┴─┐┌────────────┐ │           │ZZ(0.625)              │                           ┌─┴─┐      │    │  └───┘  │        │           │           │            │                          │             │           │           └───┘ │           ┌─┴─┐                            │      ┌─┴─┐    └───┘  │                │    │  
q_3 -> 3 ┤ Rz(-1.125) ├───────────────┼────────────────────────┼───■───────────┼────────────┼───────────■─────────────┼───┼───────────■──────────┤ X ├┤ Rz(-0.125) ├─┼───────────■───────────■───────────┼────────────■──────────────┤ X ├──────┼────┼─────────┼────■───┼───────────┼───────────┼────────────┼──────────────────────────┼────■────■───┼───────────┼─────────────■───┼───────────┤ X ├─■────────────■─────────────┼──────┤ X ├───────────┼───■────────────┼────┼──
         └────────────┘               │ZZ(1.125)             ┌─┴─┐             │            │           │             │   │           │          └───┘└────────────┘ │ZZ(1.125)              │           │ZZ(-0.625)  │ZZ(1.125)     └───┘    ┌─┴─┐  │    

In [11]:
print_circuit_info(tqc_rzz, 'TQC Rzz')

TQC Rzz has 6 qubits
TQC Rzz has 49 non-local gates and 35 non-local depth
TQC Rzz contains ['rzz', 'cx', 'rz'] gates.


In [ ]:
basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx", "swap"]
method = 'unitary'
backend_options = dict(
    method=method,
    device='CPU',
    precision='single',
    basis_gates=basis_gates
)

config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, **backend_options)
backend.set_option("n_qubits", num_physical_qubits)
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')

In [ ]:
swaps = []
for instruction in tqc.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc.swap(*swap)
tqc.save_unitary()


swaps = []
for instruction in tqc_rzz.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc_rzz.swap(*swap)
tqc_rzz.save_unitary()

In [ ]:
default = QAOAAnsatz(hamiltonian, reps=1, initial_state=QuantumCircuit(num_physical_qubits), mixer_operator=QuantumCircuit(num_physical_qubits))
t_default = transpile(default, backend, optimization_level=3)
t_default.save_unitary()
res_default = backend.run(t_default.assign_parameters([1])).result()
u_def = np.asarray(res_default.get_unitary())

In [ ]:
res_rz = backend.run(tqc).result()
u_rz = np.asarray(res_rz.get_unitary())

In [ ]:
res_rzz = backend.run(tqc_rzz).result()
u_rzz = np.asarray(res_rzz.get_unitary())

In [ ]:
np.nonzero((u_rzz - u_def) > 1e-5)

In [ ]:
n, T = 2, 2
ess = ExtendedSwapStrategy.from_grid(n, T)
max_layers = 0
router = CommutingGateRouterPrecomputeRzz(
    ess,
    max_layers=max_layers,
)

In [ ]:
from qiskit_qaoa.utils.shortest_sequence_graph_reset import labels_to_bitrows, heuristic_spanning_tree_solver

In [ ]:
labels = {0: {0,1}, 1:{0,1,2}, 2:{1,2}, 3: {0,3}}
labels_to_bitrows([0,1,2,3], labels)

In [ ]:
heuristic_spanning_tree_solver([0,1,2,3], [(0,1), (1,2), (2,3)], labels)